# **1. 손글씨 도형**

- 손으로 그린 원, 삼각형, X 이미지를 CNN으로 분류
- 데이터 구성: 학습 데이터 240장, 테스트 60장, 3개의 클래스가 균등하기 구성

In [19]:
import torch
import random
import numpy as np
import torch.nn as nn
import torch.optim as optim
import zipfile
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from pathlib import Path

In [ ]:
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [15]:
extract_dir = Path('./data')
if not extract_dir.exists():
    extract_dir.mkdir(parents=True, exist_ok=True) # parents: 상위 폴더까지 만들기   exist_ok: 이미 폴더가 존재한다면 pass

In [16]:
# 압축 풀기

with zipfile.ZipFile("./data/shape.zip", "r") as zip_ref:
    zip_ref.extractall("./data")

In [18]:
shape_root = extract_dir / 'shape'
train_dir = shape_root / 'train'
test_dir = shape_root / 'test'
print('학습 경로:', train_dir)
print('테스트 경로:', test_dir)

학습 경로: data\shape\train
테스트 경로: data\shape\test


# **2. ImageFolder**
하위 폴더 이름을 클래스 이름으로 사용하며, 알파벳 순서대로 클래스 번호를 지정함

In [22]:
# cir(0), tri(1), x(2)
raw_train = datasets.ImageFolder(train_dir)
raw_test = datasets.ImageFolder(test_dir)

print('클래스:', raw_train.classes)
print('클래스 번호:', raw_train.class_to_idx)
print('학습 데이터 이미지 수:', len(raw_train))
print('테스트 데이터 이미지 수:', len(raw_test))

클래스: ['cir', 'tri', 'x']
클래스 번호: {'cir': 0, 'tri': 1, 'x': 2}
학습 데이터 이미지 수: 240
테스트 데이터 이미지 수: 60


# **3. 전처리와 데이터 증강**
- 모든 이미지를 28*28 1채널 흑백으로 통일함
- 흰 배경에 검은 선으로 그려진 이미지를 반전하여 선 부분이 큰 값을 갖게 함
- 픽셀 값을 텐서로 바꾼 뒤, 평균 0.5, 표준편차 0.5로 정규화
- 일반화 성능을 높이기 위해 학습 데이터에만 회전/이동/크기 등을 변화

In [23]:
train_transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomInvert(p=1.0),
    transforms.RandomAffine(degrees=12, translate=(0.08, 0.08), scale=(0.90, 1.10)),
    # ToTensor(): 텐서형으로 변환. 0~1사이로 정규화
    transforms.ToTensor(),
    # 중앙값을 0, 범위를 -1~1로 조절
    transforms.Normalize(mean=(0.5,), std=(0.5,))
])

eval_transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomInvert(p=1.0),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,))
])